# Playwright

`Playwright` is a Python library that simulates the use of web pages in a very versatile way. It is super useful for scraping web pages with `JavaScript` and is ideal for obtaining thousands of records.\
To start using it, we need to do two things:\
1. Install the Playwright Chromium, which will allow us to create an environment for web scraping.\
2. Install the library itself.\
If we encounter an error, we can try uninstalling Chromium and reinstalling it, as this can be one of the main reasons for errors.

```zsh
# Install Playwright
uv add playwright

# Install Playwright Chromium
playwright install chromium
```

## Synchronous version

If we want to create a synchronous bot we use the following code. This code will navegate to the page and extract the content with BeautifulSoup.

```python
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright


def main() -> None:
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page()
        page.goto("https://books.toscrape.com/catalogue/page-1.html")
        html = page.content()
        soup = BeautifulSoup(html, "lxml")

        books = soup.find_all("li", class_="col-xs-6 col-sm-4 col-md-3 col-lg-3")

        for item in books:
            title = item.h3.a.attrs["title"]
            print(title)


main()
```

`Playwright sync_api` can't be used in `Jupyter Notebook`, because it is not designed to work in an asynchronous environment. If you want to use `Playwright in Jupyter Notebook`, you should use the `asynchronous version of Playwright`, which is compatible with Jupyter's event loop.

## Asynchronous version

To create an asynchronous bot, we need to use the `async` and `await` keywords. This code will navegate to the page and extract the content with BeautifulSoup.

```python
import asyncio

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


async def main() -> None:
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto("https://books.toscrape.com/catalogue/page-1.html")
        html = await page.content()
        soup = BeautifulSoup(html, "lxml")

        books = soup.find_all("li", class_="col-xs-6 col-sm-4 col-md-3 col-lg-3")

        for item in books:
            title = item.h3.a.attrs["title"]
            print(title)


asyncio.run(main())
```

In [ ]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


async def main() -> None:
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto("https://books.toscrape.com/catalogue/page-1.html")
        html = await page.content()
        soup = BeautifulSoup(html, "lxml")

        books = soup.find_all("li", class_="col-xs-6 col-sm-4 col-md-3 col-lg-3")

        for item in books:
            title = item.h3.a.attrs["title"]
            print(title)


await main()

NotImplementedError: 

## Playwright Methods

In this section, we will explore the different methods available in Playwright for browser management, interaction, and advanced features.

### Browser Management & Contexts

#### Launching the Browser
When launching a browser, you can configure several parameters:
*   `headless`: Run without UI (True/False). Default is True (fast), but False is better for debugging.
*   `devtools`: Open DevTools automatically.
*   `channel`: Specify 'chrome', 'msedge', etc.
*   `proxy`: Configure a proxy server.
*   `ignore_default_args`: Ignore default arguments (e.g., to disable the automation banner).
*   `timeout`: Set a timeout for browser launch.
*   `env`: Set environment variables for the browser process.

**Python Example:**
```python
browser = await p.chromium.launch(headless=False, slow_mo=500)
```

#### Contexts
A **Context** is like an incognito window. It isolates cookies, local storage, and session data. You can create multiple contexts in a single browser instance to simulate different users.

**Python Example:**
```python
context1 = await browser.new_context()
context2 = await browser.new_context() # Completely isolated from context1
```

#### Persistent Context
If you need to save cookies/session data to disk (so you don't log in every time), use `launch_persistent_context`.

**Python Example:**
```python
context = await p.chromium.launch_persistent_context(user_data_dir="./my-user-data", headless=False)
```

In [ ]:
import asyncio

from playwright.async_api import async_playwright


async def browser_management_example() -> None:
    async with async_playwright() as p:
        # Launch with Headless=False to see the browser
        browser = await p.chromium.launch(headless=False, slow_mo=500)
        print("Browser launched")

        # Create two isolated contexts (like 2 incognito windows)
        ctx1 = await browser.new_context()
        ctx2 = await browser.new_context()
        print("Contexts created")

        page1 = await ctx1.new_page()
        await page1.goto("https://ident.me")  # Check IP/Session

        await browser.close()


await browser_management_example()

## Navigation, Selectors, and Interactions

### Navigation & Waiting
*   `page.goto("url")`: Navigate to a URL.
*   `page.wait_for_load_state("networkidle")`: Wait until network is idle (no pending requests).
*   `page.reload()`: Refresh the page.

### Selectors (Locators vs QuerySelector)
Playwright encourages using **Locators** (`page.locator()`) over `query_selector`. Locators are stricter and auto-wait for elements to be actionable.

**Examples:**
*   **CSS**: `page.locator(".btn-primary")`
*   **ID**: `page.locator("#username")`
*   **Text**: `page.locator("text=Login")` or `page.get_by_text("Login")`
*   **XPath**: `page.locator("//div[@class='menu']")`

### Interactions
Once you have minimal selector, you can interact:
1.  **Click**: `await page.click("text=Login")` or `await locator.click()`
2.  **Fill Form**: `await page.fill("#username", "myuser")`
3.  **Type (Human-like)**: `await page.type("#search", "hello world", delay=100)`
4.  **Keys**: `await page.keyboard.press("Enter")`

**Python Example:**
```python
await page.locator("#search_query_top").fill("Faded Short Sleeve T-shirts")
await page.locator("button[name='submit_search']").click()
```

In [ ]:
from playwright.async_api import async_playwright


async def interaction_example() -> None:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        # Navigation
        await page.goto("http://quotes.toscrape.com/login")

        # Selectors (using id and css)
        username_field = page.locator("#username")
        password_field = page.locator("#password")
        login_btn = page.locator("input[type='submit']")

        # Interactions
        await username_field.fill("myuser")
        await password_field.fill("mypassword")

        # Click and wait for navigation
        # Ideally use page.expect_navigation() context manager but simple await works if it triggers one
        await login_btn.click()

        # Check if login failed/succeeded (by checking url or an element)
        print(f"Current URL: {page.url}")

        await browser.close()


await interaction_example()

## 3. Data Extraction, Waiting & Advanced JS

### Getting Content
We can extract data from elements using these methods:
*   `inner_text()`: Visible text only.
*   `text_content()`: All text (including hidden).
*   `get_attribute("href")`: Get attribute value.

### Evaluating JavaScript
Sometimes we need to run custom JS on the page:
*   `page.evaluate("() => document.title")`: Returns the result of the JS execution.
*   **Listeners**: Detect event handlers (e.g., specific `on` events).

### Cookies
To manage sessions or bypass logins:
*   `context.cookies()`: Get all cookies.
*   `context.add_cookies([...])`: Set cookies.

### Waiting
Playwright automatically waits for elements, but explicit waits are sometimes needed:
*   `page.wait_for_selector(".my-element")`
*   `page.wait_for_timeout(3000)` (Hard wait - try to avoid this)
*   `page.wait_for_url("**/dashboard")`

In [ ]:
from playwright.async_api import async_playwright


async def advanced_example():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(user_agent="MyBot/1.0") # Set custom User-Agent
        page = await context.new_page()
        await page.goto("http://books.toscrape.com/")

        # 1. Evaluate JS
        title = await page.evaluate("() => document.title")
        print(f"Title via JS: {title}")

        # 2. Extract Data
        # Using locator and looping
        elems = await page.locator("article.product_pod h3 a").all()
        for i, el in enumerate(elems[:3]):
            book_title = await el.get_attribute("title")
            print(f"Book {i+1}: {book_title}")

        # 3. Cookies
        cookies = await context.cookies()
        print(f"Number of cookies: {len(cookies)}")
        
        # 4. Listeners (Advanced JS)
        # Get elements with 'onclick' attribute (example conceptual)
        listeners = await page.evaluate("""
            () => {
                const e = document.body;
                return Object.keys(e).filter(k => k.startsWith('on') && e[k] !== null);
            }
        """)
        print(f"Body listeners: {listeners}")

        await browser.close()

await advanced_example()

## 4. Advanced Integrations & Tools

### Using Proxies (e.g., Bright Data)
Playwright makes it easy to route traffic through proxies.
Example configuration:
```python
proxy = {
    "server": "http://brd.superproxy.io:22225",
    "username": "...",
    "password": "..."
}
browser = await p.chromium.launch(proxy=proxy)
```

### Automation Code Generator (`Codegen`)
Playwright includes a tool that records your interactions and generates code.
To use it, run this command in your terminal:
```bash
playwright codegen wikipedia.org
```
This will open a browser and an inspector window. As you refine your actions, code (Python, JS, etc.) is generated in real-time.

### Connecting to Remote Browsers (CDP)
Often used with cloud scraping providers (like Bright Data / Scraping Browser) to offload the heavy lifting (captchas, etc).

```python
# Typically connects via WebSocket
browser = await p.chromium.connect_over_cdp(ws_endpoint)
```